# Gold: patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.patients`, LEFT JOINed to hospitals, insurance providers and per-patient lab result aggregates. Full rebuild via CREATE OR REPLACE TABLE AS SELECT.

In [0]:

# ============================================================
# NOTEBOOK: NB_patient_360
# TARGET: databrickscaresync.gold.patient_360
# GRAIN: One row per patient
# LOAD: FULL REBUILD
# ============================================================

import re

# STEP 1: CONFIGURATION

dbutils.widgets.text("catalog", "databrickscaresync")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog}")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

print(f"[SOURCE] {source_schema}")
print(f"[TARGET] {target_table}")


# STEP 2: SOURCE VALIDATION

required_columns = {
    "patients": {
        "patient_id", "first_name", "last_name",
        "date_of_birth", "gender", "city", "state",
        "registered_hospital_id", "insurance_provider_id"
    },
    "hospitals": {
        "hospital_id", "hospital_name"
    },
    "insurance_providers": {
        "insurance_provider_id", "provider_name"
    }
}

# lab_results is checked separately: if its schema is wrong, lab metrics
# fall back to 0 / NULL instead of failing the whole build.
lab_required = {"lab_result_id", "patient_id", "result_status", "test_date"}

for table_name, required in required_columns.items():
    full_name = f"{source_schema}.{table_name}"

    if not spark.catalog.tableExists(full_name):
        raise ValueError(f"Missing source table: {full_name}")

    available = set(spark.table(full_name).columns)
    missing = required - available

    if missing:
        raise ValueError(
            f"Schema mismatch in {full_name}. "
            f"Missing columns: {sorted(missing)}"
        )

lab_table = f"{source_schema}.lab_results"
lab_available = (
    set(spark.table(lab_table).columns) if spark.catalog.tableExists(lab_table) else set()
)
lab_missing = lab_required - lab_available
use_lab_results = not lab_missing

if use_lab_results:
    print("[VALIDATION PASSED] lab_results schema OK - lab metrics will be calculated")
else:
    print(
        f"[WARNING] {lab_table} is missing {sorted(lab_missing)}. "
        "Building patient_360 WITHOUT lab metrics (total/abnormal/critical = 0, last_test_date = NULL)."
    )

print("[VALIDATION PASSED]")


# STEP 3: BUILD PATIENT 360

if use_lab_results:
    lab_agg_sql = f"""
    SELECT
        patient_id,
        COUNT(*) AS total_tests,
        COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical') AS critical_test_count,
        MAX(CAST(test_date AS DATE)) AS last_test_date
    FROM {source_schema}.lab_results
    GROUP BY patient_id"""
else:
    # Empty, correctly typed placeholder so the LEFT JOIN yields 0 / NULL
    lab_agg_sql = """
    SELECT
        CAST(NULL AS INT) AS patient_id,
        CAST(NULL AS BIGINT) AS total_tests,
        CAST(NULL AS BIGINT) AS abnormal_test_count,
        CAST(NULL AS BIGINT) AS critical_test_count,
        CAST(NULL AS DATE) AS last_test_date
    WHERE 1 = 0"""

select_sql = f"""
WITH lab_agg AS ({lab_agg_sql}
),
-- Silver dimensions can contain repeated loads of the same key;
-- keep the latest row per key so joins never multiply patients
hospitals_dedup AS (
    SELECT * FROM {source_schema}.hospitals
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY hospital_id
        ORDER BY ingestion_timestamp DESC, updated_at DESC
    ) = 1
),
insurance_providers_dedup AS (
    SELECT * FROM {source_schema}.insurance_providers
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY insurance_provider_id
        ORDER BY ingestion_timestamp DESC, updated_at DESC
    ) = 1
)

SELECT
    CAST(p.patient_id AS INT) AS patient_id,
    CAST(p.first_name AS STRING) AS first_name,
    CAST(p.last_name AS STRING) AS last_name,
    CAST(p.date_of_birth AS DATE) AS date_of_birth,

    CAST(
        FLOOR(
            DATEDIFF(
                current_date(),
                CAST(p.date_of_birth AS DATE)
            ) / 365.25
        ) AS INT
    ) AS age,

    CAST(p.gender AS STRING) AS gender,
    CAST(p.city AS STRING) AS city,
    CAST(p.state AS STRING) AS state,

    CAST(p.registered_hospital_id AS INT) AS hospital_id,
    CAST(h.hospital_name AS STRING) AS hospital_name,

    CAST(p.insurance_provider_id AS INT) AS insurance_provider_id,
    CAST(ip.provider_name AS STRING) AS insurance_provider_name,

    CAST(COALESCE(l.total_tests, 0) AS INT) AS total_tests,
    CAST(COALESCE(l.abnormal_test_count, 0) AS INT)
        AS abnormal_test_count,
    CAST(COALESCE(l.critical_test_count, 0) AS INT)
        AS critical_test_count,

    CAST(l.last_test_date AS DATE) AS last_test_date

FROM {source_schema}.patients p

LEFT JOIN hospitals_dedup h
    ON p.registered_hospital_id = h.hospital_id

LEFT JOIN insurance_providers_dedup ip
    ON p.insurance_provider_id = ip.insurance_provider_id

LEFT JOIN lab_agg l
    ON p.patient_id = l.patient_id
"""


# STEP 4: FULL LOAD

spark.sql(
    f"CREATE OR REPLACE TABLE {target_table} AS {select_sql}"
)

print(f"[DONE] Rebuilt {target_table}")


# STEP 5: DATA QUALITY VALIDATION

check = spark.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT patient_id) AS distinct_patients,
    SUM(total_tests) AS total_tests
FROM {target_table}
""").first()

print(check)

assert check.total_rows == check.distinct_patients, (
    "Duplicate patient records detected"
)

source_patients = spark.table(f"{source_schema}.patients").count()
assert check.total_rows == source_patients, (
    f"Gold rows ({check.total_rows}) differ from silver.patients ({source_patients})"
)

display(
    spark.table(target_table)
    .orderBy("patient_id")
    .limit(20)
)


In [0]:
# Inspect the current Bronze table mappings
for table_name in [
    "patients",
    "hospitals",
    "doctors",
    "insurance_providers",
    "lab_results"
]:
    table = f"databrickscaresync.bronze.{table_name}"
    print(f"\n========== {table_name} ==========")
    print(spark.table(table).columns)
    print("Rows:", spark.table(table).count())